In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OCQ_J.xpt"
csv_file = DATA_DIR / "OCQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 OCQ_J XPT -> CSV")
print("=" * 90)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OCD150",
    "OCQ180",
    "OCQ210",
    "OCQ260",
    "OCD270",
    "OCQ670",
    "OCQ380",
    "OCD390G",
    "OCD395"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "OCQ_J column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 6161:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,161 rows x 10 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached OCQ_J shows that every
# populated variable is whole-number.
#
# These include:
#
#     SEQN
#     questionnaire response codes
#     work hours
#     employment-duration months
#
# No genuine fractional/continuous decimal variable occurs
# in this particular OCQ_J file.
#
# IMPORTANT:
#
# This applies ONLY to OCQ_J.
#
# Genuine decimals in Dietary, Laboratory, Examination,
# weights, exposures, nutrients, etc. must NOT be rounded
# or converted simply to remove ".0".
#

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT/pandas decoding artifact:
#
#     5.397605346934028e-79
#
# This value must NEVER be globally replaced without
# validating the exact variable/count against CDC.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 10 OCQ_J variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Attached OCQ_J.xpt:
#
#     OCD270 = 98 tiny values
#
# CDC OCQ_J:
#
# OCD270
#     0 = Less than 1 month
#     Count = 98
#
# Therefore the only permitted zero restoration is:
#
#     OCD270 tiny -> 0
#

EXPECTED_TINY_COUNTS = {
    "OCD270": 98
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated OCQ_J attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 98:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros "
        "already existed before correction."
    )


print(
    "\nPASS: Exact OCD270 98-value "
    "tiny-zero pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED OCD270 ZEROS
# ============================================================

#
# DO NOT USE:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#
# Only OCD270 has independently validated zero values.
#

ocd270_tiny_mask = (
    df["OCD270"]
    .eq(TINY_VALUE)
)


correction_count = int(
    ocd270_tiny_mask.sum()
)


if correction_count != 98:

    raise ValueError(
        "Unexpected OCD270 tiny-value count."
    )


df.loc[
    ocd270_tiny_mask,
    "OCD270"
] = 0


print("\n--- VALIDATED ZERO RESTORATION ---")

print(
    "OCD270 restored to zero:",
    correction_count
)


if int(
    df["OCD270"]
    .eq(0)
    .sum()
) != 98:

    raise ValueError(
        "OCD270 zero restoration failed."
    )


print(
    "PASS: 98 validated OCD270 zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE
# ============================================================

#
# expected_df represents exactly what the corrected source
# should contain.
#
# The ONLY permitted numeric modification is:
#
#     OCD270 tiny -> 0
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["OCD270"]
    .eq(TINY_VALUE),
    "OCD270"
] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE converting anything to Int64.
#
# np.round() below is ONLY a comparison test.
#
# It does NOT modify df.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:
        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables found in OCQ_J.\n"
        "These variables will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 11. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93705:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93705-102956 validated."
)


# ============================================================
# 12. CONVERT PROVEN WHOLE-NUMBER VARIABLES TO Int64
# ============================================================

#
# Every variable was checked BEFORE this conversion.
#
# NO:
#
#     df.round()
#
# is used.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 10 OCQ_J variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 13. EXACT FREQUENCY CHECK HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )

    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        actual_missing
    )

    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. CDC CHECK - OCD150
# ============================================================

check_frequency(
    "OCD150",
    {
        1: 3141,
        2: 122,
        3: 271,
        4: 2625,
        9: 2
    },
    0
)


# ============================================================
# 15. CDC RANGE CHECK HELPER
# ============================================================

def check_range_summary(
    column,
    lower,
    upper,
    expected_range_count,
    special_counts,
    expected_missing
):

    values = df[column]

    range_count = int(
        values.between(
            lower,
            upper,
            inclusive="both"
        )
        .sum()
    )

    actual_missing = int(
        values.isna()
        .sum()
    )

    print(
        f"\n--- CDC RANGE CHECK: {column} ---"
    )

    print(
        f"Range {lower}-{upper}:",
        range_count
    )

    if range_count != expected_range_count:

        raise ValueError(
            f"{column} range count differs from CDC.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {range_count:,}"
        )

    for code, expected_count in special_counts.items():

        actual_count = int(
            values.eq(code)
            .sum()
        )

        print(
            f"Code {code}:",
            actual_count
        )

        if actual_count != expected_count:

            raise ValueError(
                f"{column} code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} CDC range summary matches."
    )


# ============================================================
# 16. CDC CHECK - OCQ180
# ============================================================

#
# Hours worked last week.
#
# CDC:
#
# 6-78      = 3,039
# 5         = 49
# 80+       = 49
# Refused   = 1
# Don't know = 3
# Missing   = 3,020
#

check_range_summary(
    "OCQ180",
    6,
    78,
    3039,
    {
        5: 49,
        80: 49,
        77777: 1,
        99999: 3
    },
    3020
)


# ============================================================
# 17. CDC CHECK - OCQ210
# ============================================================

check_frequency(
    "OCQ210",
    {
        1: 367,
        2: 645,
        9: 2
    },
    5147
)


# ============================================================
# 18. CDC CHECK - OCQ260
# ============================================================

check_frequency(
    "OCQ260",
    {
        1: 2376,
        2: 93,
        3: 196,
        4: 190,
        5: 346,
        6: 25,
        77: 6,
        99: 31
    },
    2898
)


# ============================================================
# 19. CDC CHECK - OCD270
# ============================================================

#
# Number of months working in main job.
#
# CDC:
#
# 1-468     = 3,116
# 0         = 98
# 480+      = 38
# Refused   = 7
# Don't know = 4
# Missing   = 2,898
#

check_range_summary(
    "OCD270",
    1,
    468,
    3116,
    {
        0: 98,
        480: 38,
        77777: 7,
        99999: 4
    },
    2898
)


# ============================================================
# 20. CDC CHECK - OCQ670
# ============================================================

check_frequency(
    "OCQ670",
    {
        1: 1168,
        2: 423,
        3: 483,
        5: 1185,
        7: 2,
        9: 1
    },
    2899
)


# ============================================================
# 21. CDC CHECK - OCQ380
# ============================================================

check_frequency(
    "OCQ380",
    {
        1: 324,
        2: 294,
        3: 1189,
        4: 201,
        5: 35,
        6: 370,
        7: 211,
        77: 1
    },
    3536
)


# ============================================================
# 22. CDC CHECK - OCD390G
# ============================================================

check_frequency(
    "OCD390G",
    {
        1: 3739,
        2: 2015,
        3: 46,
        4: 354,
        7: 4,
        9: 3
    },
    0
)


# ============================================================
# 23. CDC CHECK - OCD395
# ============================================================

#
# Duration of longest job.
#
# CDC:
#
# 6-588     = 3,578
# 5         = 145
# 600+      = 53
# Refused   = 2
# Don't know = 7
# Missing   = 2,376
#

check_range_summary(
    "OCD395",
    6,
    588,
    3578,
    {
        5: 145,
        600: 53,
        77777: 2,
        99999: 7
    },
    2376
)


# ============================================================
# 24. ALLOWED CODE CHECK
# ============================================================

ALLOWED_VALUES = {

    "OCD150": {
        1, 2, 3, 4, 7, 9
    },

    "OCQ210": {
        1, 2, 7, 9
    },

    "OCQ260": {
        1, 2, 3, 4, 5, 6, 77, 99
    },

    "OCQ670": {
        1, 2, 3, 5, 7, 9
    },

    "OCQ380": {
        1, 2, 3, 4, 5, 6, 7, 77, 99
    },

    "OCD390G": {
        1, 2, 3, 4, 7, 9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    unexpected = observed - allowed

    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )

    print(
        f"PASS: {col} contains only "
        "CDC-documented codes."
    )


# ============================================================
# 25. ROUTING CHECK HELPER
# ============================================================

def check_route_mask(
    description,
    expected_mask,
    child_column
):

    expected = (
        expected_mask
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    present = (
        df[child_column]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )

    mismatch_count = int(
        np.sum(
            expected != present
        )
    )

    print(
        f"\n--- ROUTING: {description} ---"
    )

    print(
        "Expected populated:",
        int(expected.sum())
    )

    print(
        "Actually populated:",
        int(present.sum())
    )

    print(
        "Routing mismatches:",
        mismatch_count
    )

    if mismatch_count != 0:

        raise ValueError(
            f"Routing validation failed: "
            f"{description}"
        )

    print(
        "PASS: Routing validated."
    )


# ============================================================
# 26. CURRENT-JOB ROUTING
# ============================================================

#
# Current-job information exists when OCD150 is:
#
#     1 = Working
#     2 = Has job but not at work
#
# There are 3,263 such records.
#

current_job_mask = (
    df["OCD150"]
    .isin(
        [
            1,
            2
        ]
    )
)


check_route_mask(
    "OCD150 in {1,2} -> OCQ260",
    current_job_mask,
    "OCQ260"
)


check_route_mask(
    "OCD150 in {1,2} -> OCD270",
    current_job_mask,
    "OCD270"
)


# ============================================================
# 27. NOT-WORKING ROUTING
# ============================================================

#
# OCQ380 is populated exactly for:
#
#     OCD150 = 4
#
# Count = 2,625
#

check_route_mask(
    "OCD150=4 -> OCQ380",
    df["OCD150"].eq(4),
    "OCQ380"
)


# ============================================================
# 28. LONGEST-JOB ROUTING
# ============================================================

#
# OCD395 is populated when OCD390G indicates:
#
#     1 = occupation entered
#     3 = armed forces
#
# 3,739 + 46 = 3,785 records.
#

check_route_mask(
    "OCD390G in {1,3} -> OCD395",
    df["OCD390G"].isin(
        [
            1,
            3
        ]
    ),
    "OCD395"
)


# ============================================================
# 29. FINAL ZERO PATTERN BEFORE EXPORT
# ============================================================

#
# The ONLY expected numeric zero values are:
#
#     OCD270 = 98
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs from "
        "validated OCQ_J correction pattern.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern contains exactly "
    "98 OCD270 zeros."
)


# ============================================================
# 30. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Only permitted numeric modification:
#
#     OCD270 tiny -> 0
#
# All other numeric values must remain identical.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 10 variables match exact "
    "expected corrected XPT values."
)


# ============================================================
# 31. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position mismatches before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 32. EXPORT CSV
# ============================================================

#
# IMPORTANT FIDELITY POLICY:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
# NO blanket decimal formatting.
#
# OCQ_J itself contains no genuine fractional variables.
#
# However, this export policy remains important for your
# broader NHANES pipeline so that genuine Dietary,
# Laboratory, Examination, etc. decimals remain AS IS.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 33. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 34. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 10 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 35. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- CSV MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "during export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 36. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 37. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 38. RAW CSV .0 CHECK
# ============================================================

#
# OCQ_J has been proven to contain no genuine decimals.
#
# Therefore all ten variables may safely be checked
# for unnecessary trailing ".0".
#
# IMPORTANT:
#
# Do NOT apply this global all-column check to a mixed
# file containing genuine decimal variables.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any OCQ_J field."
)


# ============================================================
# 39. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0)
        .sum()
    )

    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "98 OCD270 zeros."
)


# ============================================================
# 40. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]

    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 41. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93705:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN 93705-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 42. FINAL COMPLETE SOURCE -> CSV FIDELITY
# ============================================================

final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    final_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves exact corrected "
    "numeric values for all 10 variables."
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "OCQ_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Questionnaire - Occupation"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)

print(
    "OCD270 zero restorations:",
    correction_count
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "OCD270 <1 month:",
    int(
        df["OCD270"]
        .eq(0)
        .sum()
    )
)

print(
    "Current-job records:",
    int(
        df["OCD150"]
        .isin(
            [
                1,
                2
            ]
        )
        .sum()
    )
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0': "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 OCQ_J XPT -> CSV
Rows: 6,161
Columns: 10
PASS: Structure = 6,161 rows x 10 columns.

--- FIELD TYPE PLAN ---
Genuine decimal variables: []
Whole-number/code variables: 10

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 98

Tiny values by variable:
OCD270    98

PASS: Exact OCD270 98-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
OCD270 restored to zero: 98
PASS: 98 validated OCD270 zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
OCD150: 0 fractional observations
OCQ180: 0 fractional observations
OCQ210: 0 fractional observations
OCQ260: 0 fractional observations
OCD270: 0 fractional observations
OCQ670: 0 fractional observations
OCQ380: 0 fractional observations
OCD390G: 0 fractional observations
OCD395: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 93705.0
Ma